In [1]:
import os
import json
import mne
import numpy as np
import pandas as pd
from scipy.signal import resample
from natsort import natsorted
import warnings
warnings.filterwarnings("ignore")
from collections import defaultdict
import scipy.io as sio
import mne

In [8]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 100  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 50    # e.g. 50 means 50% overlap for SEG_LEN=100

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L100'

In [3]:
root = 'ADFSU/EEG_data'
# Note that the eyes_open data for healthy with patient ID 5 is empty in the raw data. For simplicity, we just manually copy the eyes_closed data for this patient and paste it as eyes_open data to avoid any empty check in the code.

In [ ]:
labels = np.empty((92,2), dtype='int32')
for i in range(92):
    if i < 80:
        labels[i, 0] = 1
    else:
        labels[i, 0] = 0
    labels[i, 1] = i+1
print(labels)

In [4]:
def load_selected_channels(folder_path, channel_list):
    channels_data = []
    for channel in channel_list:
        file_path = os.path.join(folder_path, f"{channel}.txt")
        
        with open(file_path, 'r') as f:
            channel_data = np.array([float(line.strip()) for line in f])
            channels_data.append(channel_data)
    
    data_array = np.stack(channels_data, axis=0).T
    return data_array

In [5]:
def resample_time_series(data, original_fs, target_fs):
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    
    resampled_data = np.zeros((new_length, C))
    for i in range(C):
        resampled_data[:, i] = resample(data[:, i], new_length)
        
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [23]:
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
input_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 
                'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
n_channels = len(input_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP)


global_sub = 1
for file in os.listdir(root):
    print(f"{file}-----------------------------\n")
    path = os.path.join(root, file)
    eye_closed_list = []
    eye_open_list = []
    sub = None  # local subject ID within each class
    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        subject_path_list = []
        sub = global_sub
        for subject in os.listdir(state_path):
            subject_path = os.path.join(state_path, subject)
            subject_path_list.append(subject_path)
        subject_path_list = natsorted(subject_path_list)
        for subject_path in subject_path_list:
            data = load_selected_channels(subject_path, input_channels)
            T_raw = data.shape[0]
            original_fs = 128  # original sampling rate in Hz
            for fs in SAMPLE_RATE_LIST:
                # resampled length if each trial
                T_res = int(T_raw * fs / original_fs)
                # compute number of segments
                n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                subject_segment_counts[global_sub][fs] += n_seg
                N_total += n_seg
                print("Subject ID:", sub, "Label", file, "State:", state)
                print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
            print("-------------------------------------\n")
            sub += 1
    global_sub = global_sub + sub - 1
    print("-------------------------------------\n")

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 100 OVERLAP = 50 STEP = 50
AD-----------------------------

Subject ID: 1 Label AD State: Eyes_closed
fs=100Hz: T_res=800, STEP=50, n_seg=15
Subject ID: 1 Label AD State: Eyes_closed
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject ID: 2 Label AD State: Eyes_closed
fs=100Hz: T_res=800, STEP=50, n_seg=15
Subject ID: 2 Label AD State: Eyes_closed
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject ID: 3 Label AD State: Eyes_closed
fs=100Hz: T_res=800, STEP=50, n_seg=15
Subject ID: 3 Label AD State: Eyes_closed
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject ID: 4 Label AD State: Eyes_closed
fs=100Hz: T_res=800, STEP=50, n_seg=15
Subject ID: 4 Label AD State: Eyes_closed
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject ID: 5 Label AD State: Eyes_closed
fs=100Hz: T_res=800, STEP=50, n_seg=15
Subject ID: 5 Label AD State: Eyes_closed
fs=50Hz: T_

In [29]:
output_root = os.path.join('Processed', sub_folder_path, 'ADFSU')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L100\ADFSU\X.dat
y path: Processed\L100\ADFSU\y.dat


## PASS 2: Process and store segments into memmap

In [30]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
global_sub = 1
for file in os.listdir(root):
    print(f"{file}-----------------------------\n")
    label = 1 if file == 'AD' else 0
    path = os.path.join(root, file)
    eye_closed_list = []
    eye_open_list = []
    sub = None  # local subject ID within each class
    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        subject_path_list = []
        sub = global_sub
        for subject in os.listdir(state_path):
            subject_path = os.path.join(state_path, subject)
            subject_path_list.append(subject_path)
        subject_path_list = natsorted(subject_path_list)
        for subject_path in subject_path_list:
            data_raw = load_selected_channels(subject_path, input_channels)
            T_raw = data_raw.shape[0]
            original_fs = 128  # original sampling rate in Hz
            total_seconds_all += data_raw.shape[0] / original_fs
            for fs in SAMPLE_RATE_LIST:
                # resample to target fs
                data = resample_time_series(data_raw, original_fs, fs)
                T_res, _ = data.shape
                print(f"fs={fs}Hz: resampled shape={data.shape}")
    
                # overlapped sliding window with fixed STEP (in timestamps)
                starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                print("Subject ID:", sub, "Label", file, "State:", state)
                print(f"fs={fs}Hz: segments={len(starts)}")
    
                for s in starts:
                    if cur >= N_total:
                        raise RuntimeError("Exceeded predicted N_total.")
    
                    X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                    y_mm[cur, 0] = float(label)       # label
                    y_mm[cur, 1] = float(sub)         # subject ID
                    y_mm[cur, 2] = float(fs)          # sample rate (scale)
                    cur += 1
            print("-------------------------------------\n")
            sub += 1
    global_sub = global_sub + sub - 1
    print("-------------------------------------\n")

total_minutes_all = total_seconds_all / 60.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total minutes across all subjects: {total_minutes_all:.2f} minutes")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

AD-----------------------------

fs=100Hz: resampled shape=(800, 19)
Subject ID: 1 Label AD State: Eyes_closed
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
Subject ID: 1 Label AD State: Eyes_closed
fs=50Hz: segments=7
-------------------------------------

fs=100Hz: resampled shape=(800, 19)
Subject ID: 2 Label AD State: Eyes_closed
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
Subject ID: 2 Label AD State: Eyes_closed
fs=50Hz: segments=7
-------------------------------------

fs=100Hz: resampled shape=(800, 19)
Subject ID: 3 Label AD State: Eyes_closed
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
Subject ID: 3 Label AD State: Eyes_closed
fs=50Hz: segments=7
-------------------------------------

fs=100Hz: resampled shape=(800, 19)
Subject ID: 4 Label AD State: Eyes_closed
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
Subject ID: 4 Label AD State: Eyes_closed
fs=50Hz: segments=7
-------------------------------------

fs=100Hz: resampled sha

## Load and check the processed data

In [31]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 4048
  T = 100
  C = 19
  X_path = Processed\L100\ADFSU\X.dat
  y_path = Processed\L100\ADFSU\y.dat
-------------------------------------
Subject ID 001: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 002: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 003: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 004: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 005: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 006: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 007: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 008: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 009: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 010: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 011: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 012: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 013: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 014: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 015: X shape=(44, 100, 19), y shape=(44, 3)
Subject ID 016: X shape=(